In [1]:
#bert_ag_news.pth
import os
import torch

os.environ['http_proxy'] = 'http://127.0.0.1:7890'
os.environ['https_proxy'] = 'http://127.0.0.1:7890'
os.environ['all_proxy'] = 'socks5://127.0.0.1:7890'

os.environ['HF_HOME'] = '/home/bobsun/cambrige/MedMinist/hugginface'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/home/bobsun/cambrige/MedMinist/hugginface'
from datasets import load_dataset

# 加载 AG News 数据集
dataset = load_dataset('ag_news')

from transformers import BertTokenizer
# 加载 BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [5]:
from transformers import BertTokenizer
from datasets import load_dataset, DatasetDict, Dataset, IterableDatasetDict, IterableDataset
from typing import Union
# 加载 BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def preprocess_function(examples):
    tokenized = tokenizer(examples['text'], padding='max_length', truncation=True, max_length=71)
    return {
        'input_ids': tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask'],
        'labels': examples['label']
    }

tokenized_datasets: Union[DatasetDict, Dataset, IterableDatasetDict, IterableDataset] = dataset.map(preprocess_function, batched=True, remove_columns=["text"])

from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader

train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

# 定义 DataCollator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding='max_length', max_length=71, return_tensors='pt')

# 创建 DataLoader，指定 collate_fn
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=data_collator)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=data_collator)



In [10]:
import importlib
import Transformer.DEN_BERT

importlib.reload(Transformer.DEN_BERT)


config = Transformer.DEN_BERT.BertConfig(
    num_labels=4  # 根据 AG News 数据集的标签数调整
)

state_dict = torch.load('den_ag_news.pth', map_location='cpu')

# 实例化自定义模型
model = Transformer.DEN_BERT.CustomBertForSequenceClassification(config)

# 加载调整后的 state_dict
model.load_state_dict(state_dict, strict=True)  # 使用 strict=True 确保所有参数匹配
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
test(model,test_dataloader)

100%|██████████| 238/238 [00:03<00:00, 67.14it/s]

Validation Accuracy: 93.46%


In [12]:
# 模拟 input_ids：随机整数在1到vocab_size-1之间（0通常是[PAD] token）
input_ids = torch.randint(1, 30522, (1, 70))

# 模拟 attention_mask：全1表示所有token都需要被注意
attention_mask = torch.ones((1, 70), dtype=torch.long)

# 将张量移动到设备
input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)

# 前向传播
with torch.no_grad():
    logits = model(input_ids, attention_mask=attention_mask)


In [87]:
import numpy as np
from matplotlib import cm, colors
def saliency_to_latex(tokens, saliency, cmap='viridis'):
    """
    将tokens和saliency scores转换为带有背景颜色和透明度的LaTeX代码，使用RGB颜色模型。

    Args:
        tokens (list): 分词后的token列表。
        saliency (numpy array): 对应每个token的归一化saliency scores（0到1）。
        cmap (str): 使用的colormap名称（默认是'viridis'）。

    Returns:
        str: 生成的LaTeX代码。
    """
    # 选择colormap
    colormap = cm.get_cmap(cmap)

    # 归一化saliency scores
    norm = colors.Normalize(vmin=0, vmax=1)

    latex_tokens = []
    for token, score in zip(tokens, saliency):
        # 获取颜色
        rgba = colormap(norm(score))
        # 提取RGB值（忽略alpha通道）
        r, g, b, _ = rgba
        # 计算透明度，映射到0.4到1.0之间
        opacity = 0.4 + 0.6 * score
        # 生成带有RGB背景颜色和透明度的TikZ命令
        latex_token = (
            f'\\begin{{tikzpicture}}[baseline=(word.base)]\n'
            f'  \\node[fill={{rgb,1:red,{r:.2f}; green,{g:.2f}; blue,{b:.2f}}}, '
            f'opacity={opacity:.2f}, inner sep=1pt, rounded corners=2pt] (word) '
            f'{{\\strut {token}}};\n'
            f'\\end{{tikzpicture}}'
        )
        latex_tokens.append(latex_token)

    # 将所有token连接为一行
    latex_line = ' '.join(latex_tokens)
    return latex_line
def bert_cam_one_layer(layer_idx,class_idx):
    layers = model.encoder.layer
    activation = layers[layer_idx].deb.activation.squeeze(0).cpu().detach().numpy() # [70,768]

    W = layers[layer_idx].deb.W.weight
    pred_fc = layers[layer_idx].deb.pred_fc.weight
    weight = pred_fc @ W

    neuron = weight[class_idx].cpu().detach().numpy() # ndarray: [768] 单个神经元的权重, 代表70(max_length = 70) 个768维向量,求平均以后对该类别做的贡献

    a = neuron * activation
    sailency = np.sum(a, axis=1)
    return sailency

def get_all_sailency(model,input_ids,attention_mask):
    logits = model(input_ids, attention_mask=attention_mask)
    print("distribution:",torch.nn.Softmax(dim=1)(logits))
    predictions = torch.argmax(logits, dim=-1)
    sailencies = []
    for i in range(12):
        sailency = bert_cam_one_layer(i,predictions.item())
        sailencies.append(sailency)
    sailencies = np.array(sailencies)
    return sailencies

def encode_text(text):
    return tokenizer(text, padding='max_length', truncation=True, max_length=71, return_tensors='pt')


import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import matplotlib.cm as cm
from IPython.display import display, HTML
from matplotlib.cm import get_cmap
def render_text_with_saliency(tokens, saliency, cmap='Reds'):
    """
    Render text with tokens colored according to saliency scores.

    Args:
        tokens (list): List of tokens.
        saliency (numpy array): Array of saliency scores corresponding to tokens.
        cmap (str): Colormap to use.

    Returns:
        HTML object displaying colored text.
    """
    # 归一化saliency scores到[0,1]
    norm = Normalize(vmin=np.min(saliency), vmax=np.max(saliency))
    cmap = cm.get_cmap(cmap)

    # 将saliency转换为颜色
    cmap = get_cmap('viridis')

    # 将每个值映射为颜色
    colors = [plt.cm.colors.rgb2hex(cmap(norm(score))) for score in saliency]

    # 生成带有颜色的HTML文本
    colored_tokens = [
        f'<span style="background-color:{color}; padding:2px; border-radius:3px;">{token}</span>'
        for token, color in zip(tokens, colors)
    ]

    # 连接tokens
    html_text = ' '.join(colored_tokens)

    return HTML(html_text)




# text = "Michael Phelps won the gold medal in the 400 individual medley and set a world record in a time of 4 minutes 8.26 seconds."
# text = "TGn Sync Proposes New WLAN Standard	The battle over home entertainment networking is heating up as a coalition proposes yet another standard for the IEEE #39;s consideration."
text = "LONDON (Reuters) - Oil prices surged to a new high of \$47 a  barrel on Wednesday after a new threat by rebel militia against  Iraqi oil facilities and as the United States said inflation  had stayed in check despite rising energy costs."
inputs = encode_text(text)
input_ids = inputs['input_ids'].to(device)[:,1:]
attention_mask = inputs['attention_mask'].to(device)[:,1:]

result = get_all_sailency(model,input_ids,attention_mask)
a = result
result = result[0:3]
result = np.sum(result,axis=0)

attention_mask = attention_mask.squeeze().detach().cpu().numpy()
zero_indices = np.where(attention_mask == 0)[0]

if zero_indices.size > 0:
    first_zero_index = zero_indices[0]
    result = result[:first_zero_index]


result_min = np.min(result)
result_max = np.max(result)

result_norm = (result - result_min) / (result_max - result_min)

tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
# 打印每个 token 及其对应的索引
# for idx, token in enumerate(tokens):
#     print(f"{idx}: {token}")
latex = saliency_to_latex(tokens, result_norm)
# print(latex)
display(render_text_with_saliency(tokens, result_norm))

distribution: tensor([[5.1330e-02, 9.4784e-01, 5.4138e-04, 2.8930e-04]], device='cuda:0',
       grad_fn=<SoftmaxBackward0>)


/tmp/ipykernel_3690233/798673543.py:16: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  colormap = cm.get_cmap(cmap)
/tmp/ipykernel_3690233/798673543.py:91: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = cm.get_cmap(cmap)
/tmp/ipykernel_3690233/798673543.py:94: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = get_cmap('viridis')


In [4]:
from transformers import BertTokenizer
# 加载 BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def preprocess_function(examples):
    tokenized = tokenizer(examples['text'], padding='max_length', truncation=True, max_length=70)
    return {
        'input_ids': tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask'],
        'labels': examples['label']
    }

tokenized_datasets = dataset.map(preprocess_function, batched=True, remove_columns=["text"])
test_dataset = tokenized_datasets['test']

from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader


# 定义 DataCollator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding='max_length', max_length=70, return_tensors='pt')

test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=data_collator)


100%|██████████| 238/238 [00:03<00:00, 71.71it/s]

Validation Accuracy: 94.61%


In [9]:
from tqdm import tqdm


def test(model,test_dataloader):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in tqdm(test_dataloader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask,)
            logits = outputs

            predictions = torch.argmax(logits, dim=-1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    accuracy = correct / total
    print(f"Validation Accuracy: {accuracy * 100:.2f}%")